# PMT Baseline Diagnostics

Carga desde el cache generado por `pmt_preprocess.py` y muestra diagnósticos de baseline.  
**Prerequisito:** haber ejecutado `pmt_preprocess.py --run <RUN>` al menos una vez.

Contenido:
1. Timestamp check
2. Distribución de residuales de baseline (todos los canales)
3. Residuales para un canal concreto
4. Distribución del RMS de baseline
5. Visor de una waveform individual
6. Visor de waveforms más anómalas (ipywidgets)
7. Heatmap de todos los canales (plotly)

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from waffles.data_classes.Waveform import Waveform
from waffles.data_classes.WaveformSet import WaveformSet
from waffles.data_classes.UniqueChannel import UniqueChannel
from waffles.input_output.pickle_hdf5_reader import WaveformSet_from_hdf5_pickle
from waffles.np02_utils.AutoMap import ordered_modules_pmt
from waffles.np02_utils.PlotUtils import plot_detectors

In [ ]:
# --- Configuración del run ---
run     = 43363
dettype = 'pmt'

cache_file = f"cache/wfset_run{run:06d}_{dettype}_ana.hdf5"
wfset_triggered_all = WaveformSet_from_hdf5_pickle(cache_file)
print(f"Cargadas {len(wfset_triggered_all.waveforms)} waveforms desde {cache_file}")

## 1. Timestamp check

In [ ]:
diffvalues = [wf.daq_window_timestamp - wf.timestamp for wf in wfset_triggered_all.waveforms]
dmax = np.max(diffvalues)
dmin = np.min(diffvalues)
print(f"max={dmax}  min={dmin}  range={( dmax - dmin)*16e-9:.3e} s")

plt.figure(figsize=(8, 4))
plt.hist(diffvalues, bins=np.linspace(-50, 50, 100))
plt.xlabel("daq_window_timestamp - timestamp [ticks]")
plt.ylabel("Count")
plt.title(f"Run {run} — timestamp offset distribution")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Distribución de residuales de baseline (todos los canales)

In [ ]:
analysis_label = "std"

baselines_by_channel = defaultdict(list)
for wf in wfset_triggered_all.waveforms:
    baselines_by_channel[(wf.endpoint, wf.channel)].append(
        wf.analyses[analysis_label].result["baseline"]
    )

mean_baseline_by_channel = {
    ch: np.mean(vals) for ch, vals in baselines_by_channel.items()
}

baseline_residuals = np.array([
    wf.analyses[analysis_label].result["baseline"] - mean_baseline_by_channel[(wf.endpoint, wf.channel)]
    for wf in wfset_triggered_all.waveforms
])

plt.figure(figsize=(8, 5))
plt.hist(baseline_residuals, bins=100, histtype="step", linewidth=1.8)
plt.xlabel("Baseline - media de canal [ADC]")
plt.ylabel("Número de waveforms")
plt.title("Distribución de fluctuaciones de baseline")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Residuales de baseline para un canal concreto

In [ ]:
target_ep = 110
target_ch = 16

residuals_ch = [
    wf.analyses["std"].result["baseline"] - mean_baseline_by_channel[(target_ep, target_ch)]
    for wf in wfset_triggered_all.waveforms
    if wf.endpoint == target_ep and wf.channel == target_ch
]

plt.figure(figsize=(8, 5))
plt.hist(residuals_ch, bins=60, histtype="step", linewidth=1.8)
plt.xlabel("Baseline - media de canal [ADC]")
plt.ylabel("Número de waveforms")
plt.title(f"Baseline residuals — endpoint {target_ep}, channel {target_ch}")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Distribución del RMS de baseline

In [ ]:
baseline_start  = 0
baseline_finish = 65
x_range         = (0, 100)

baseline_rms_values = np.array([
    np.sqrt(np.mean(
        (wf.adcs[baseline_start:baseline_finish] - wf.analyses["std"].result["baseline"]) ** 2
    ))
    for wf in wfset_triggered_all.waveforms
])

plt.figure(figsize=(8, 5))
plt.hist(baseline_rms_values, bins=50, range=x_range, histtype="step", linewidth=1.8)
plt.xlim(x_range)
plt.xlabel("Baseline RMS [ADC]")
plt.ylabel("Número de waveforms")
plt.title("Distribución del RMS de baseline — todos los canales")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Visor de una waveform individual

In [ ]:
selected_endpoint = 110
selected_channel  = 40
waveform_index    = 3506
subtract_baseline = True

wfset_ch = WaveformSet.from_filtered_WaveformSet(
    wfset_triggered_all,
    lambda wf: wf.endpoint == selected_endpoint and wf.channel == selected_channel,
    show_progress=False,
)

if len(wfset_ch.waveforms) == 0:
    raise ValueError(f"No waveforms for ep={selected_endpoint}, ch={selected_channel}")
if waveform_index >= len(wfset_ch.waveforms):
    raise IndexError(f"waveform_index={waveform_index} pero solo hay {len(wfset_ch.waveforms)}")

wf       = wfset_ch.waveforms[waveform_index]
baseline = wf.analyses["std"].result["baseline"]
y_plot   = wf.adcs - baseline if subtract_baseline else wf.adcs
times_ns = np.arange(len(y_plot)) * 16

plt.figure(figsize=(10, 5))
plt.plot(times_ns, y_plot, color="black", linewidth=1.2)
if subtract_baseline:
    plt.axhline(0, color="red", linestyle="--", linewidth=1, alpha=0.8, label="baseline subtracted")
    plt.ylabel("Amplitude - baseline [ADC]")
else:
    plt.axhline(baseline, color="red", linestyle="--", linewidth=1, alpha=0.8, label=f"baseline = {baseline:.2f}")
    plt.ylabel("ADC")
plt.xlabel("Time [ns]")
plt.title(f"ep {selected_endpoint} | ch {selected_channel} | wf {waveform_index}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Visor de waveforms más anómalas (ipywidgets)

Necesita `ipywidgets`. Ejecutar la celda de abajo directamente.

In [ ]:
from ipywidgets import interact, IntSlider
import time

selected_endpoint        = 110
selected_channel         = 14
analysis_label           = "std"
baseline_window          = slice(0, 65)
peak_window              = slice(60, 100)
integral_window          = slice(60, 160)
n_anomalous_to_keep_all  = 10000

t0 = time.time()
selected_waveforms_all = [
    wf for wf in wfset_triggered_all.waveforms
    if wf.endpoint == selected_endpoint and wf.channel == selected_channel
]
if not selected_waveforms_all:
    raise ValueError(f"No waveforms for ep={selected_endpoint}, ch={selected_channel}")

def robust_z(values):
    values = np.asarray(values, dtype=float)
    median = np.median(values)
    mad    = np.median(np.abs(values - median))
    scale  = 1.4826 * mad if mad > 0 else 1.0
    return (values - median) / scale

metrics = []
for idx, wf in enumerate(selected_waveforms_all):
    baseline    = wf.analyses[analysis_label].result["baseline"]
    y           = wf.adcs - baseline
    pre_rms     = np.std(y[baseline_window])
    peak_tick   = peak_window.start + np.argmax(y[peak_window])
    peak_amp    = float(y[peak_tick])
    integral    = float(np.sum(y[integral_window]))
    metrics.append({"idx": idx, "baseline": baseline, "pre_rms": pre_rms,
                    "peak_tick": peak_tick, "peak_amp": peak_amp, "integral": integral})

anomaly_scores = (
    np.abs(robust_z([m["pre_rms"]   for m in metrics])) +
    np.abs(robust_z([m["peak_amp"]  for m in metrics])) +
    np.abs(robust_z([m["integral"]  for m in metrics]))
)
sorted_indices = np.argsort(anomaly_scores)[::-1]
top_n = min(n_anomalous_to_keep_all, len(sorted_indices))
top_waveforms = [selected_waveforms_all[i] for i in sorted_indices[:top_n]]
print(f"Seleccionadas {top_n} waveforms más anómalas en {time.time()-t0:.1f} s")

@interact(idx=IntSlider(min=0, max=top_n - 1, step=1, value=0, description="Rank"))
def show_anomalous(idx):
    wf       = top_waveforms[idx]
    baseline = wf.analyses[analysis_label].result["baseline"]
    y        = np.asarray(wf.adcs, dtype=float) - baseline
    plt.figure(figsize=(10, 4))
    plt.plot(np.arange(len(y)) * 16, y)
    plt.xlabel("Time [ns]")
    plt.ylabel("ADC - baseline")
    plt.title(f"Anomaly rank {idx} | score={anomaly_scores[sorted_indices[idx]]:.2f}")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

## 7. Heatmap de todos los canales (plotly)

In [ ]:
argsheat = dict(
    mode="heatmap",
    analysis_label="std",
    adc_range_above_baseline=1000,
    adc_range_below_baseline=-250,
    adc_bins=400,
    time_bins=wfset_triggered_all.points_per_wf // 2,
    filtering=4,
    share_y_scale=True,
    share_x_scale=True,
    wfs_per_axes=5000,
    zlog=True,
    width=1600,
    height=1200,
)
detector = ordered_modules_pmt
plot_detectors(wfset_triggered_all, detector, **argsheat)